# Prepare Gold Labels After Snorkel

This notebook is intended to run after the Snorkel weak-supervision step.

It reads the weak-labeled dataset, injects authoritative hardcoded gold labels, optionally adds development-only dummy labels, and saves a final dataset that is ready for both Stage 1 and Stage 2.

The notebook creates three outputs:

- `gold_labels_clean.csv`: authoritative hardcoded labels only
- `gold_labels_debug_augmented.csv`: hardcoded labels plus development-only dummy labels
- `contract_with_features_labeled_with_gold.csv`: weak-labeled dataset with gold labels joined on


> **Why this notebook fills the gold-label table.** Stage 2 MAML and FOMAML
> need at least `k_pos` positive and `k_neg` negative gold-labeled contracts
> per meta-*train* department, otherwise `filter_valid_departments` empties
> the task set and the outer loop silently produces zero predictions. The
> policy below guarantees every department ends with **≥5 yes / ≥5 no** and
> that **Logistics** (the meta-test target) ends with **≥10 yes / ≥10 no**.
> Re-run this notebook end-to-end whenever the hardcoded dictionary changes,
> whenever the feature table is regenerated, or whenever Stage 2 fails with
> empty `predictions.csv` artefacts.


## 1. Environment Setup

The notebook is intended to be run from the `notebooks/` directory. It expects that the Snorkel notebook has already been executed and that `contract_with_features_labeled.csv` is available in `Data/processed`.

This file is used as the base input because it already contains the weak label signal `renegotiation_prob`. The final output from this notebook is the dataset that Stage 1 and Stage 2 should read.


In [1]:
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

PROJECT_ROOT = Path("..").resolve()
DATA_PROCESSED = PROJECT_ROOT / "Data" / "processed"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

FEATURE_DATA_PATH = DATA_PROCESSED / "contract_with_features_labeled.csv"

if not FEATURE_DATA_PATH.exists():
    raise FileNotFoundError(
        "Could not find contract_with_features_labeled.csv in Data/processed. "
        "Run the Snorkel notebook first."
    )

df_features = pd.read_csv(FEATURE_DATA_PATH, low_memory=False)

print("Loaded weak-labeled dataset:", FEATURE_DATA_PATH.name)
print("Shape:", df_features.shape)
print(
    "Unique contracts:",
    df_features["contract_id"].nunique() if "contract_id" in df_features.columns else "contract_id missing",
)
print(
    "Departments:",
    df_features["department"].nunique() if "department" in df_features.columns else "department missing",
)
print(
    "Has renegotiation_prob:",
    "renegotiation_prob" in df_features.columns,
)
display(df_features.head())


Loaded weak-labeled dataset: contract_with_features_labeled.csv
Shape: (9201, 165)
Unique contracts: 2209
Departments: 14
Has renegotiation_prob: True


,contract_id,contract_name,contract_status,contract_owner_cost_centre,terminated,term_type,start_date,expiration_date,supplier_id,supplier_number,supplier_display_name,payment_terms,incoterms,contract_currency_code,contract_value,contract_value_dkk,contract_type,nn_contract_type,contract_commodity,team,unit,company_country,start_year,expiry_year,open_ended_contract,end_year,start_year_capped,observation_year,contract_age_years,expiry_pressure_bucket,department_from_cost_center,department,moodys_bvd_id,fin_moodys_company_name,fin_closing_date,fin_created_at_utc,fin_Status,fin_implied_rating,fin_risk_level,fin_financial_level,fin_output_text,fin_Implied_rating - original,fin_Number_of_months,fin_Total_shareholders'_funds_and_liabilitiesth_DKK,fin_profit_margin,fin_EBITDA_margin,fin_ebit_margin,fin_current_ratio,fin_gearing,fin_Cash_ratio,fin_long_term_gearing,fin_Total_liabilities_Equity_ratio,fin_short_term_liabilities_equity_ratio,fin_interest_coverage_ratio,fin_solvency_ratio_asset_based,fin_debt_asset_ratio,fin_roe_net_income,fin_roa_net_income,fin_Net_assets_turnover,fin_Number_of_employees,fin_financial_risk_score,esg_esg_overall,esg_esg_industry_adjusted,esg_env_score,esg_env_weight,esg_social_score,esg_social_weight,esg_gov_score,esg_gov_weight,esg_industry_min,esg_industry_max,news_article_count,news_negative_count,news_positive_count,news_neutral_count,news_avg_sentiment_score,news_avg_relevance_score,news_max_relevance_score,news_negative_ratio,news_has_high_relevance_negative_news,Company name Latin alphabet,avg_vol,vol_stability_score,vol_shock_ratio,vol_trend_slope,Risk level,market_cap_volatility,supplier_sector,moodys_risk_rating,Price_trends_52 weeks_%,Earnings_per_share_DKK,Shares outstanding,Current_market_capitalisation_DKK,avg_closing_price,price_volatility_score,price_trend_slope,Risk level_stock_closing_price,company_country_clean,Country,Code,LPI_Score,Customs,Infrastructure,International_Shipments,Tracking_Tracing,Timeliness,PPI_Value,gold_y,gold_department,fin_implied_rating_missing_flag,fin_financial_level_missing_flag,fin_current_ratio_missing_flag,fin_interest_coverage_ratio_missing_flag,fin_ebit_margin_missing_flag,fin_implied_rating_ordinal,fin_flag_moderate_or_worse_rating,fin_flag_risk_take_caution_or_worse,fin_flag_risk_do_not_source,fin_flag_financial_risk_score_high,fin_flag_financial_risk_score_very_high,fin_flag_liquidity_stress,fin_flag_severe_liquidity_stress,fin_flag_strong_liquidity,fin_flag_gearing_high,fin_flag_long_term_gearing_high,fin_flag_short_term_gearing_high,fin_flag_debt_asset_high,fin_flag_long_term_liab_equity_high,fin_flag_short_term_liab_equity_high,fin_flag_interest_coverage_stress,fin_flag_interest_coverage_weak,fin_flag_low_solvency,fin_flag_very_low_solvency,fin_flag_negative_ebit_margin,fin_flag_negative_roe,fin_flag_negative_roa,fin_total_stress_flags,fin_flag_multiple_financial_stress_signals,fin_flag_severe_financial_stress,esg_below_industry_min,supplier_is_publicly_listed,market_flag_high_volume_shock,market_flag_high_market_cap_volatility,market_flag_negative_volume_trend,market_flag_negative_price_trend,market_flag_negative_52w_price_trend,market_flag_high_beta,market_flag_negative_eps,market_flag_stock_price_take_caution_or_worse,market_log_vol_shock_ratio,lpi_relative_risk,is_old_and_near_expiry,esg_row_missing_pct,news_row_missing_pct,lpi_below_supplier_median,years_to_expiry_capped,contracts_per_supplier,financial_data_available,esg_data_available,news_data_available,market_data_available,has_environmental_appendix,contract_name_lower,renegotiation_prob,target_renegotiate
0,9675,Bioreliance_Master_2018_MSA,published,7756.0,0,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,BIORELIANCE LIMITED INVITROGEN BIOSERVICES,F030 Invoice Date + 30 days,DDP Delivered duty paid,GBP,0.0,0.0,Production,Master Service Agreement,Outsourced GMP Laboratory Analysis,"Quality, Production Services & Supplies",SSIMS,DENMARK,2018,2027,0,2025,2018,2018,0,5y_pl

## 2. Hardcoded Gold Label Dictionary

This dictionary is the manual holder for authoritative labels. Add or remove rows directly in the relevant department list.

Each row should contain:
- `contract_id`
- `contract_number`
- `gold_y`

The department is inherited from the dictionary key, so it does not need to be repeated inside each row.


In [2]:
gold_labels_by_department = {
    "Packaging Material": [
        {'contract_id': '193', 'contract_number': '193', 'gold_y': 1},
        {'contract_id': '213', 'contract_number': '213', 'gold_y': 1},
        {'contract_id': '190', 'contract_number': '190', 'gold_y': 1},
        {'contract_id': '1265', 'contract_number': '1265', 'gold_y': 1},
        {'contract_id': '1245', 'contract_number': '1245', 'gold_y': 1},
        {'contract_id': '1256', 'contract_number': '1256', 'gold_y': 1},
        {'contract_id': '248', 'contract_number': '248', 'gold_y': 0},
        {'contract_id': '250', 'contract_number': '250', 'gold_y': 0},
    ],
    "Devices & Needles": [
    ],
    "Drug Product Outsourcing": [
    ],
    "Drug Substance Outsourcing": [
    ],
    "Raw Materials & Energy": [
    ],
    "Quality, Production Services & Supplies": [
    ],
    "Bioprocessing and Excipients": [
        {'contract_id': '504361', 'contract_number': '504149', 'gold_y': 1},
        {'contract_id': '613927', 'contract_number': '612555', 'gold_y': 1},
        {'contract_id': '345318', 'contract_number': '346312', 'gold_y': 0},
        {'contract_id': '1877', 'contract_number': '1877', 'gold_y': 0},
    ],
    "Bioprocessing & Raw Materials": [
    ],
    "Alliance, Acquisitions & PPM CoE": [
    ],
    "Global Strategic Outsourcing & Devices": [
        {'contract_id': '4821', 'contract_number': '4822', 'gold_y': 1},
        {'contract_id': '557', 'contract_number': '557', 'gold_y': 1},
        {'contract_id': '565', 'contract_number': '565', 'gold_y': 1},
        {'contract_id': '483149', 'contract_number': '483178', 'gold_y': 0},
        {'contract_id': '25777', 'contract_number': '25777', 'gold_y': 0},
    ],
    "Strategic Sourcing US Hub": [
    ],
    "Strategy, Sourcing & Negotiation CoE": [
    ],
    "HI Warehouse Expansion Program": [
    ],
    "Aseptic Production": [
        {'contract_id': '577893', 'contract_number': '577893', 'gold_y': 0},
        {'contract_id': '370849', 'contract_number': '370849', 'gold_y': 0},
        {'contract_id': '595207', 'contract_number': '595207', 'gold_y': 0},
    ],
    "Logistics": [
        {'contract_id': '192312', 'contract_number': '192325', 'gold_y': 1},
        {'contract_id': '541883', 'contract_number': '541269', 'gold_y': 1},
        {'contract_id': '582913', 'contract_number': '581732', 'gold_y': 1},
        {'contract_id': '569028', 'contract_number': '568080', 'gold_y': 1},
        {'contract_id': '470427', 'contract_number': '470608', 'gold_y': 1},
        {'contract_id': '530290', 'contract_number': '529779', 'gold_y': 1},
        {'contract_id': '533151', 'contract_number': '532634', 'gold_y': 0},
        {'contract_id': '550918', 'contract_number': '550237', 'gold_y': 0},
        {'contract_id': '525275', 'contract_number': '524758', 'gold_y': 0},
        {'contract_id': '335471', 'contract_number': '336520', 'gold_y': 0},
        {'contract_id': '607710', 'contract_number': '606340', 'gold_y': 0},
        {'contract_id': '536891', 'contract_number': '536319', 'gold_y': 0},
        {'contract_id': '599608', 'contract_number': '598142', 'gold_y': 0},
    ],
}


## 3. Dummy Fill Parameters

The next cell controls the optional development-only augmentation logic.

The rule is:

- every eligible department should end with at least `TARGET_YES` positive labels,
- every eligible department should end with at least `TARGET_NO` negative labels,
- unless the department is explicitly excluded from dummy filling,
- and unless the department does not contain enough unique contracts to support the requested total.

If a department already satisfies the target, nothing is added. If not, the notebook injects only the missing amount as `DUMMY_FILLER` rows from unlabeled contracts in that same department.


In [3]:
USE_DUMMY_FILL = True

# Global defaults — every eligible department ends with at least these counts.
TARGET_YES = 5
TARGET_NO = 5

# Per-department overrides. Departments not listed here use the global defaults.
# Logistics is the meta-test target in Stage 2, so it gets a higher quota to
# support k-shot sweeps up to k=10 and to stabilise target-episode evaluation.
DEPARTMENT_TARGET_OVERRIDES = {
    "Logistics": {"yes": 10, "no": 10},
}

RANDOM_SEED = 42

# Departments listed here will never receive dummy labels.
# Keep the names exactly as they appear in the dataset.
EXCLUDE_DUMMY_FILL_DEPARTMENTS = [
    # "Strategy, Sourcing & Negotiation CoE",
    # "Alliance, Acquisitions & PPM CoE",
]

# Random injection (per user spec). Set SCORE_COL = "renegotiation_prob" if
# you want score-ordered injection (highest scores -> yes, lowest -> no) instead
# of uniform random sampling from the unlabeled pool.
SCORE_COL = None


## 4. Flatten and Standardize the Hardcoded Dictionary

This section converts the department-level dictionary into a row-based table, adds metadata, and standardizes identifiers and labels.


In [4]:
all_departments = sorted(df_features["department"].dropna().astype(str).str.strip().unique().tolist())

def flatten_gold_dict(gold_dict: dict) -> list[dict]:
    rows = []
    today = str(date.today())

    for department, items in gold_dict.items():
        for item in items:
            row = item.copy()
            row["department"] = department
            row["label_source"] = row.get("label_source", "manual_hardcoded")
            row["label_date"] = row.get("label_date", today)
            row["notes"] = row.get("notes", "")
            rows.append(row)

    return rows

gold_label_records = flatten_gold_dict(gold_labels_by_department)

df_gold_clean = pd.DataFrame(gold_label_records)

if df_gold_clean.empty:
    df_gold_clean = pd.DataFrame(
        columns=[
            "contract_id",
            "contract_number",
            "department",
            "gold_y",
            "label_source",
            "label_date",
            "notes",
        ]
    )

required_cols = [
    "contract_id",
    "contract_number",
    "department",
    "gold_y",
    "label_source",
    "label_date",
    "notes",
]

for col in required_cols:
    if col not in df_gold_clean.columns:
        df_gold_clean[col] = pd.NA

df_gold_clean = df_gold_clean[required_cols].copy()

for col in ["contract_id", "contract_number", "department"]:
    df_gold_clean[col] = (
        df_gold_clean[col]
        .astype("string")
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )

df_gold_clean["gold_y"] = pd.to_numeric(df_gold_clean["gold_y"], errors="coerce")

invalid_labels = df_gold_clean[~df_gold_clean["gold_y"].isin([0, 1]) & df_gold_clean["gold_y"].notna()]
if not invalid_labels.empty:
    raise ValueError("Invalid gold_y values found. Only 0 and 1 are allowed.")

df_gold_clean["gold_y"] = df_gold_clean["gold_y"].astype("Int64")
df_gold_clean["label_source"] = df_gold_clean["label_source"].fillna("manual_hardcoded")
df_gold_clean["label_date"] = df_gold_clean["label_date"].fillna(str(date.today()))
df_gold_clean["notes"] = df_gold_clean["notes"].fillna("")

# Remove exact duplicate label rows.
df_gold_clean = df_gold_clean.drop_duplicates().copy()

print("Authoritative hardcoded gold labels:", len(df_gold_clean))
display(df_gold_clean.head(20))


Authoritative hardcoded gold labels: 33


,contract_id,contract_number,department,gold_y,label_source,label_date,notes
0,193,193,Packaging Material,1,manual_hardcoded,2026-04-18,
1,213,213,Packaging Material,1,manual_hardcoded,2026-04-18,
2,190,190,Packaging Material,1,manual_hardcoded,2026-04-18,
3,1265,1265,Packaging Material,1,manual_hardcoded,2026-04-18,
4,1245,1245,Packaging Material,1,manual_hardcoded,2026-04-18,
5,1256,1256,Packaging Material,1,manual_hardcoded,2026-04-18,
6,248,248,Packaging Material,0,manual_hardcoded,2026-04-18,
7,250,250,Packaging Material,0,manual_hardcoded,2026-04-18,
8,504361,504149,Bioprocessing and Excipients,1,manual_hardcoded,2026-04-18,
9,613927,612555,Bioprocessing and Excipients,1,manual_hardcoded,2026-04-18,


## 5. Development-Only Dummy Injection

This section optionally injects dummy labels so that each eligible department approaches the requested minimum number of positive and negative labels. Existing hardcoded labels are never overwritten.

The core rule is:

- `yes_to_add = max(0, TARGET_YES - existing_yes)`
- `no_to_add = max(0, TARGET_NO - existing_no)`

Additional safeguards are applied:

- departments listed in `EXCLUDE_DUMMY_FILL_DEPARTMENTS` are skipped,
- departments with missing names are skipped,
- no department can receive more labels than there are unique unlabeled contracts available,
- if a department has fewer unique contracts than `TARGET_YES + TARGET_NO`, the notebook injects as many labels as possible and reports the final shortfall.

All injected rows are marked as `DUMMY_FILLER`.


In [5]:
df_features_ids = df_features.copy()

for col in ["contract_id", "contract_number", "department"]:
    if col in df_features_ids.columns:
        df_features_ids[col] = (
            df_features_ids[col]
            .astype("string")
            .str.replace(r"\.0$", "", regex=True)
            .str.strip()
        )

# Remove rows with missing contract ids because they cannot be labeled reliably.
df_features_ids = df_features_ids[df_features_ids["contract_id"].notna()].copy()

def sample_candidates_for_label(
    df_department: pd.DataFrame,
    n_needed: int,
    label_value: int,
    score_col: str,
    rng: np.random.Generator,
) -> pd.DataFrame:
    if n_needed <= 0 or df_department.empty:
        return df_department.iloc[0:0].copy()

    candidates = df_department.copy()

    if score_col in candidates.columns and candidates[score_col].notna().any():
        ascending = True if label_value == 0 else False
        candidates = candidates.sort_values(score_col, ascending=ascending)
        return candidates.head(min(n_needed, len(candidates))).copy()

    sample_n = min(n_needed, len(candidates))
    sampled_idx = rng.choice(candidates.index.to_numpy(), size=sample_n, replace=False)
    return candidates.loc[sampled_idx].copy()

rng = np.random.default_rng(RANDOM_SEED)

dummy_rows = []
dummy_fill_diagnostics = []

eligible_departments = [
    dept for dept in all_departments
    if pd.notna(dept) and str(dept).strip() != ""
]

if USE_DUMMY_FILL:
    existing_ids = set(df_gold_clean["contract_id"].dropna().astype(str).tolist())

    unique_contract_counts = (
        df_features_ids.groupby("department")["contract_id"]
        .nunique()
        .to_dict()
    )

    for department in eligible_departments:
        if department in EXCLUDE_DUMMY_FILL_DEPARTMENTS:
            dummy_fill_diagnostics.append({
                "department": department,
                "status": "excluded",
                "unique_contracts": int(unique_contract_counts.get(department, 0)),
                "manual_yes": int((df_gold_clean.loc[df_gold_clean["department"] == department, "gold_y"] == 1).sum()),
                "manual_no": int((df_gold_clean.loc[df_gold_clean["department"] == department, "gold_y"] == 0).sum()),
                "dummy_yes_added": 0,
                "dummy_no_added": 0,
            })
            continue

        dept_clean = df_gold_clean[df_gold_clean["department"] == department].copy()

        existing_yes = int((dept_clean["gold_y"] == 1).sum())
        existing_no = int((dept_clean["gold_y"] == 0).sum())

        dept_override = DEPARTMENT_TARGET_OVERRIDES.get(department, {})
        target_yes_dept = int(dept_override.get("yes", TARGET_YES))
        target_no_dept = int(dept_override.get("no", TARGET_NO))

        yes_to_add = max(0, target_yes_dept - existing_yes)
        no_to_add = max(0, target_no_dept - existing_no)

        dept_candidates = df_features_ids[
            (df_features_ids["department"] == department)
            & (~df_features_ids["contract_id"].astype(str).isin(existing_ids))
        ].copy()

        available_unique = int(dept_candidates["contract_id"].nunique())

        total_needed = yes_to_add + no_to_add
        if available_unique < total_needed:
            alloc_yes = yes_to_add
            alloc_no = no_to_add

            while alloc_yes + alloc_no > available_unique:
                yes_gap_ratio = alloc_yes / max(1, target_yes_dept)
                no_gap_ratio = alloc_no / max(1, target_no_dept)

                if yes_gap_ratio >= no_gap_ratio and alloc_yes > 0:
                    alloc_yes -= 1
                elif alloc_no > 0:
                    alloc_no -= 1
                elif alloc_yes > 0:
                    alloc_yes -= 1
                else:
                    break

            yes_to_add = alloc_yes
            no_to_add = alloc_no

        yes_candidates = sample_candidates_for_label(
            df_department=dept_candidates,
            n_needed=yes_to_add,
            label_value=1,
            score_col=SCORE_COL,
            rng=rng,
        )

        if not yes_candidates.empty:
            dept_candidates = dept_candidates[
                ~dept_candidates["contract_id"].astype(str).isin(yes_candidates["contract_id"].astype(str))
            ].copy()

        no_candidates = sample_candidates_for_label(
            df_department=dept_candidates,
            n_needed=no_to_add,
            label_value=0,
            score_col=SCORE_COL,
            rng=rng,
        )

        added_yes = 0
        added_no = 0

        for _, row in yes_candidates.iterrows():
            dummy_rows.append({
                "contract_id": str(row["contract_id"]),
                "contract_number": row.get("contract_number", pd.NA),
                "department": department,
                "gold_y": 1,
                "label_source": "DUMMY_FILLER",
                "label_date": str(date.today()),
                "notes": f"Development-only injected positive label for {department}.",
            })
            existing_ids.add(str(row["contract_id"]))
            added_yes += 1

        for _, row in no_candidates.iterrows():
            dummy_rows.append({
                "contract_id": str(row["contract_id"]),
                "contract_number": row.get("contract_number", pd.NA),
                "department": department,
                "gold_y": 0,
                "label_source": "DUMMY_FILLER",
                "label_date": str(date.today()),
                "notes": f"Development-only injected negative label for {department}.",
            })
            existing_ids.add(str(row["contract_id"]))
            added_no += 1

        dummy_fill_diagnostics.append({
            "department": department,
            "status": "filled" if (added_yes + added_no) > 0 else "unchanged",
            "unique_contracts": int(unique_contract_counts.get(department, 0)),
            "manual_yes": existing_yes,
            "manual_no": existing_no,
            "dummy_yes_added": added_yes,
            "dummy_no_added": added_no,
        })

df_gold_augmented = pd.concat(
    [df_gold_clean, pd.DataFrame(dummy_rows)],
    ignore_index=True,
).drop_duplicates(subset=["contract_id", "gold_y", "department", "label_source"])

dummy_fill_diagnostics = pd.DataFrame(dummy_fill_diagnostics)
display(dummy_fill_diagnostics.sort_values(["status", "department"]).reset_index(drop=True))


,department,status,unique_contracts,manual_yes,manual_no,dummy_yes_added,dummy_no_added
0,"Alliance, Acquisitions & PPM CoE",filled,35,0,0,5,5
1,Bioprocessing & Raw Materials,filled,113,0,0,5,5
2,Bioprocessing and Excipients,filled,59,2,2,3,3
3,Devices & Needles,filled,476,0,0,5,5
4,Drug Product Outsourcing,filled,300,0,0,5,5
5,Drug Substance Outsourcing,filled,298,0,0,5,5
6,Global Strategic Outsourcing & Devices,filled,8,3,2,2,3
7,HI Warehouse Expansion Program,filled,8,0,0,4,4
8,Logistics,filled,94,6,7,4,3
9,Packaging Material,filled,289,6,2,0,3


## 6. Save Authoritative and Augmented Gold Label Tables

The notebook saves both the clean authoritative labels and the augmented development version. This keeps the distinction between manual labels and dummy labels explicit.


In [6]:
GOLD_CLEAN_PATH = DATA_PROCESSED / "gold_labels_clean.csv"
GOLD_DEBUG_PATH = DATA_PROCESSED / "gold_labels_debug_augmented.csv"

df_gold_clean.to_csv(GOLD_CLEAN_PATH, index=False)
df_gold_augmented.to_csv(GOLD_DEBUG_PATH, index=False)

print("Saved authoritative labels to:", GOLD_CLEAN_PATH)
print("Saved augmented labels to    :", GOLD_DEBUG_PATH)


Saved authoritative labels to: /Users/Thomas/Desktop/Master Thesis/Data/processed/gold_labels_clean.csv
Saved augmented labels to    : /Users/Thomas/Desktop/Master Thesis/Data/processed/gold_labels_debug_augmented.csv


## 7. Join Labels onto the Weak-Labeled Dataset

The augmented label table is joined back onto the weak-labeled dataset so that downstream notebooks can use one consistent file containing both:

- `renegotiation_prob` from Snorkel
- `gold_y` from the hardcoded gold-label workflow

This final output is the file that Stage 1 and Stage 2 should use.


In [7]:
df_features_joined = df_features_ids.copy()

for col in ["gold_y", "gold_department", "label_source", "label_date", "notes"]:
    if col in df_features_joined.columns:
        df_features_joined = df_features_joined.drop(columns=[col])

df_gold_for_join = df_gold_augmented.copy()

join_cols = [
    "contract_id",
    "department",
    "gold_y",
    "label_source",
    "label_date",
    "notes",
]
join_cols = [col for col in join_cols if col in df_gold_for_join.columns]

df_features_with_gold = df_features_joined.merge(
    df_gold_for_join[join_cols],
    on="contract_id",
    how="left",
    suffixes=("", "_gold"),
)

if "department_gold" in df_features_with_gold.columns:
    df_features_with_gold = df_features_with_gold.rename(columns={"department_gold": "gold_department"})
else:
    df_features_with_gold["gold_department"] = pd.NA

FINAL_OUTPUT_PATH = DATA_PROCESSED / "contract_with_features_labeled_with_gold.csv"
df_features_with_gold.to_csv(FINAL_OUTPUT_PATH, index=False)

print("Saved final weak-labeled dataset with gold labels to:", FINAL_OUTPUT_PATH)
print("Shape:", df_features_with_gold.shape)
print("Rows with gold labels:", int(df_features_with_gold["gold_y"].notna().sum()))
display(df_features_with_gold.head())


Saved final weak-labeled dataset with gold labels to: /Users/Thomas/Desktop/Master Thesis/Data/processed/contract_with_features_labeled_with_gold.csv
Shape: (9201, 168)
Rows with gold labels: 700


,contract_id,contract_name,contract_status,contract_owner_cost_centre,terminated,term_type,start_date,expiration_date,supplier_id,supplier_number,supplier_display_name,payment_terms,incoterms,contract_currency_code,contract_value,contract_value_dkk,contract_type,nn_contract_type,contract_commodity,team,unit,company_country,start_year,expiry_year,open_ended_contract,end_year,start_year_capped,observation_year,contract_age_years,expiry_pressure_bucket,department_from_cost_center,department,moodys_bvd_id,fin_moodys_company_name,fin_closing_date,fin_created_at_utc,fin_Status,fin_implied_rating,fin_risk_level,fin_financial_level,fin_output_text,fin_Implied_rating - original,fin_Number_of_months,fin_Total_shareholders'_funds_and_liabilitiesth_DKK,fin_profit_margin,fin_EBITDA_margin,fin_ebit_margin,fin_current_ratio,fin_gearing,fin_Cash_ratio,fin_long_term_gearing,fin_Total_liabilities_Equity_ratio,fin_short_term_liabilities_equity_ratio,fin_interest_coverage_ratio,fin_solvency_ratio_asset_based,fin_debt_asset_ratio,fin_roe_net_income,fin_roa_net_income,fin_Net_assets_turnover,fin_Number_of_employees,fin_financial_risk_score,esg_esg_overall,esg_esg_industry_adjusted,esg_env_score,esg_env_weight,esg_social_score,esg_social_weight,esg_gov_score,esg_gov_weight,esg_industry_min,esg_industry_max,news_article_count,news_negative_count,news_positive_count,news_neutral_count,news_avg_sentiment_score,news_avg_relevance_score,news_max_relevance_score,news_negative_ratio,news_has_high_relevance_negative_news,Company name Latin alphabet,avg_vol,vol_stability_score,vol_shock_ratio,vol_trend_slope,Risk level,market_cap_volatility,supplier_sector,moodys_risk_rating,Price_trends_52 weeks_%,Earnings_per_share_DKK,Shares outstanding,Current_market_capitalisation_DKK,avg_closing_price,price_volatility_score,price_trend_slope,Risk level_stock_closing_price,company_country_clean,Country,Code,LPI_Score,Customs,Infrastructure,International_Shipments,Tracking_Tracing,Timeliness,PPI_Value,fin_implied_rating_missing_flag,fin_financial_level_missing_flag,fin_current_ratio_missing_flag,fin_interest_coverage_ratio_missing_flag,fin_ebit_margin_missing_flag,fin_implied_rating_ordinal,fin_flag_moderate_or_worse_rating,fin_flag_risk_take_caution_or_worse,fin_flag_risk_do_not_source,fin_flag_financial_risk_score_high,fin_flag_financial_risk_score_very_high,fin_flag_liquidity_stress,fin_flag_severe_liquidity_stress,fin_flag_strong_liquidity,fin_flag_gearing_high,fin_flag_long_term_gearing_high,fin_flag_short_term_gearing_high,fin_flag_debt_asset_high,fin_flag_long_term_liab_equity_high,fin_flag_short_term_liab_equity_high,fin_flag_interest_coverage_stress,fin_flag_interest_coverage_weak,fin_flag_low_solvency,fin_flag_very_low_solvency,fin_flag_negative_ebit_margin,fin_flag_negative_roe,fin_flag_negative_roa,fin_total_stress_flags,fin_flag_multiple_financial_stress_signals,fin_flag_severe_financial_stress,esg_below_industry_min,supplier_is_publicly_listed,market_flag_high_volume_shock,market_flag_high_market_cap_volatility,market_flag_negative_volume_trend,market_flag_negative_price_trend,market_flag_negative_52w_price_trend,market_flag_high_beta,market_flag_negative_eps,market_flag_stock_price_take_caution_or_worse,market_log_vol_shock_ratio,lpi_relative_risk,is_old_and_near_expiry,esg_row_missing_pct,news_row_missing_pct,lpi_below_supplier_median,years_to_expiry_capped,contracts_per_supplier,financial_data_available,esg_data_available,news_data_available,market_data_available,has_environmental_appendix,contract_name_lower,renegotiation_prob,target_renegotiate,gold_department,gold_y,label_source,label_date,notes
0,9675,Bioreliance_Master_2018_MSA,published,7756.0,0,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,BIORELIANCE LIMITED INVITROGEN BIOSERVICES,F030 Invoice Date + 30 days,DDP Delivered duty paid,GBP,0.0,0.0,Production,Master Service Agreement,Outsourced GMP Laboratory Analysis,"Quality, Production Services & Supplies",SSIMS,DENMARK,2018

## 8. Department Summary

The final summary shows, for each department:

- how many authoritative labels already existed,
- how many dummy labels were added,
- and the final class counts after augmentation.

The final modeling file produced by this notebook is `contract_with_features_labeled_with_gold.csv`, which is the dataset intended for Stage 1 and Stage 2.


In [8]:
manual_summary = (
    df_gold_clean.groupby("department", dropna=False)
    .agg(
        manual_total=("gold_y", "count"),
        manual_yes=("gold_y", "sum"),
    )
    .reset_index()
)
manual_summary["manual_no"] = manual_summary["manual_total"] - manual_summary["manual_yes"]

dummy_summary = (
    df_gold_augmented[df_gold_augmented["label_source"] == "DUMMY_FILLER"]
    .groupby("department", dropna=False)
    .agg(
        dummy_total=("gold_y", "count"),
        dummy_yes=("gold_y", "sum"),
    )
    .reset_index()
)
if dummy_summary.empty:
    dummy_summary = pd.DataFrame(columns=["department", "dummy_total", "dummy_yes", "dummy_no"])
else:
    dummy_summary["dummy_no"] = dummy_summary["dummy_total"] - dummy_summary["dummy_yes"]

summary = pd.DataFrame({"department": all_departments})
summary = summary.merge(manual_summary, on="department", how="left")
summary = summary.merge(dummy_summary, on="department", how="left")
summary = summary.fillna(0)

count_cols = ["manual_total", "manual_yes", "manual_no", "dummy_total", "dummy_yes", "dummy_no"]
summary[count_cols] = summary[count_cols].astype(int)

summary["final_yes"] = summary["manual_yes"] + summary["dummy_yes"]
summary["final_no"] = summary["manual_no"] + summary["dummy_no"]
summary["final_total"] = summary["final_yes"] + summary["final_no"]

summary = summary.sort_values(["final_total", "department"], ascending=[False, True]).reset_index(drop=True)

display(summary)
print("Total authoritative labels:", len(df_gold_clean))
print("Total dummy labels added  :", int((df_gold_augmented["label_source"] == "DUMMY_FILLER").sum()))


,department,manual_total,manual_yes,manual_no,dummy_total,dummy_yes,dummy_no,final_yes,final_no,final_total
0,Logistics,13,6,7,6,4,2,10,9,19
1,Packaging Material,8,6,2,3,0,3,6,5,11
2,"Alliance, Acquisitions & PPM CoE",0,0,0,10,5,5,5,5,10
3,Bioprocessing & Raw Materials,0,0,0,10,5,5,5,5,10
4,Bioprocessing and Excipients,4,2,2,6,3,3,5,5,10
5,Devices & Needles,0,0,0,10,5,5,5,5,10
6,Drug Product Outsourcing,0,0,0,10,5,5,5,5,10
7,Drug Substance Outsourcing,0,0,0,10,5,5,5,5,10
8,"Quality, Production Services & Supplies",0,0,0,10,5,5,5,5,10
9,Raw Materials & Energy,0,0,0,10,5,5,5,5,10


Total authoritative labels: 33
Total dummy labels added  : 105


## 9. Invariant Check — target counts met for every department

Asserts that the augmented gold-label table satisfies the per-department
quota. If any department is below target (and is not on the exclude list),
this cell raises `AssertionError` and the notebook halts. Do not save the
downstream outputs if this assertion fails — Stage 2 MAML/FOMAML will quietly
produce empty predictions on the underfilled departments.


In [9]:
invariant_summary = (
    df_gold_augmented
    .groupby("department")["gold_y"]
    .agg([("gold_yes", lambda s: int((s == 1).sum())),
          ("gold_no",  lambda s: int((s == 0).sum()))])
    .reset_index()
)

fail_rows = []
for _, r in invariant_summary.iterrows():
    dept = r["department"]
    if dept in EXCLUDE_DUMMY_FILL_DEPARTMENTS:
        continue
    override = DEPARTMENT_TARGET_OVERRIDES.get(dept, {})
    req_yes = int(override.get("yes", TARGET_YES))
    req_no  = int(override.get("no",  TARGET_NO))
    if r["gold_yes"] < req_yes or r["gold_no"] < req_no:
        fail_rows.append({
            "department": dept,
            "gold_yes":   int(r["gold_yes"]),
            "gold_no":    int(r["gold_no"]),
            "req_yes":    req_yes,
            "req_no":     req_no,
        })

display(invariant_summary.sort_values("department").reset_index(drop=True))

if fail_rows:
    import pandas as pd
    print("\nDepartments below target:")
    display(pd.DataFrame(fail_rows))
    raise AssertionError(
        f"{len(fail_rows)} department(s) do not meet the required gold-label "
        f"quota. Increase candidate pool (ingest more contracts for that dept) "
        f"or relax the quota in DEPARTMENT_TARGET_OVERRIDES / TARGET_YES / TARGET_NO."
    )

print("✓ Gold-label invariant holds for every eligible department.")


,department,gold_yes,gold_no
0,"Alliance, Acquisitions & PPM CoE",5,5
1,Aseptic Production,0,3
2,Bioprocessing & Raw Materials,5,5
3,Bioprocessing and Excipients,5,5
4,Devices & Needles,5,5
5,Drug Product Outsourcing,5,5
6,Drug Substance Outsourcing,5,5
7,Global Strategic Outsourcing & Devices,5,4
8,HI Warehouse Expansion Program,4,2
9,Logistics,10,9



Departments below target:


,department,gold_yes,gold_no,req_yes,req_no
0,Aseptic Production,0,3,5,5
1,Global Strategic Outsourcing & Devices,5,4,5,5
2,HI Warehouse Expansion Program,4,2,5,5
3,Logistics,10,9,10,10
4,Strategic Sourcing US Hub,4,4,5,5
5,"Strategy, Sourcing & Negotiation CoE",1,1,5,5


AssertionError: 6 department(s) do not meet the required gold-label quota. Increase candidate pool (ingest more contracts for that dept) or relax the quota in DEPARTMENT_TARGET_OVERRIDES / TARGET_YES / TARGET_NO.